In [1]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

ROOT_DIR = Path.cwd().parent
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

In [2]:
from src.pipeline.config import (
    TRAIN_DF_PATH,
	RANDOM_SEED
)

In [3]:
df = pd.read_parquet(TRAIN_DF_PATH, engine="pyarrow")

In [4]:
df.head()

,TransactionID,isFraud,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,card6,...,TransactionAmtMax,TransactionAmtMin,TransactionAmtStd,TransactionAmt_Z,TransactionAmt_to_Mean,DayOfWeek,HourSin,HourCos,DayOfWeekSin,DayOfWeekCos
0,2987000,0,68.5,W,13926,-1.0,150.0,discover,142.0,credit,...,317.50,68.5,176.069589,-0.707107,0.354922,1,0.0,1.0,0.781831,0.62349
1,2987001,0,29.0,W,2755,404.0,150.0,mastercard,102.0,credit,...,4592.02,29.0,614.242501,-0.407218,0.103894,1,0.0,1.0,0.781831,0.62349
2,2987002,0,59.0,W,4663,490.0,150.0,visa,166.0,debit,...,122.99,25.0,22.055991,0.297718,1.125234,1,0.0,1.0,0.781831,0.62349
3,2987003,0,50.0,W,18132,567.0,150.0,mastercard,117.0,debit,...,3190.00,6.0,248.958879,-0.301440,0.399852,1,0.0,1.0,0.781831,0.62349
4,2987004,0,50.0,H,4497,514.0,150.0,mastercard,102.0,credit,...,50.00,50.0,0.000000,0.000000,1.000000,1,0.0,1.0,0.781831,0.62349


In [5]:
X = df.drop(["DeviceInfo", "isFraud", "uid", "DayOfWeek", "TransactionID"], axis=1)
y = df["isFraud"]

In [6]:
kaggle_cat_cols = (
    ['ProductCD'] +
    [f'card{i}' for i in range(1, 7)] +
    ['addr1', 'addr2'] +
    ['P_emaildomain', 'R_emaildomain'] +
    [f'M{i}' for i in range(1, 10)] +
    ['DeviceType', 'DeviceInfo'] +
    [f'id_{i}' for i in range(12, 39)]
)

cat_features = [col for col in kaggle_cat_cols if col in X.columns]

for col in cat_features:
    X[col] = X[col].fillna('missing').astype(str)

In [7]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
	X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)

In [8]:
from catboost import CatBoostClassifier

clf = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    eval_metric='AUC',
    custom_metric=['Logloss'],
    random_seed=RANDOM_SEED,
    early_stopping_rounds=100,
    task_type='GPU',
    verbose=50
)

In [9]:
clf.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    cat_features=cat_features,
    use_best_model=True
)

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.6971510	best: 0.6971510 (0)	total: 229ms	remaining: 7m 37s
50:	test: 0.9031695	best: 0.9031695 (50)	total: 7.57s	remaining: 4m 49s
100:	test: 0.9261052	best: 0.9261052 (100)	total: 14.2s	remaining: 4m 26s
150:	test: 0.9363551	best: 0.9363764 (149)	total: 20.7s	remaining: 4m 12s
200:	test: 0.9412320	best: 0.9412320 (200)	total: 27.4s	remaining: 4m 5s
250:	test: 0.9443856	best: 0.9443856 (250)	total: 34.3s	remaining: 3m 59s
300:	test: 0.9463876	best: 0.9463876 (300)	total: 41.4s	remaining: 3m 53s
350:	test: 0.9476286	best: 0.9476286 (350)	total: 48.5s	remaining: 3m 47s
400:	test: 0.9497854	best: 0.9497854 (400)	total: 55.4s	remaining: 3m 40s
450:	test: 0.9507884	best: 0.9507897 (448)	total: 1m 2s	remaining: 3m 35s
500:	test: 0.9516739	best: 0.9516739 (500)	total: 1m 9s	remaining: 3m 28s
550:	test: 0.9527910	best: 0.9527910 (550)	total: 1m 16s	remaining: 3m 22s
600:	test: 0.9536265	best: 0.9536265 (600)	total: 1m 24s	remaining: 3m 16s
650:	test: 0.9541299	best: 0.9541299 (650)	

CatBoostClassifier(custom_metric=['Logloss'], early_stopping_rounds=100, eval_metric='AUC', iterations=2000, learning_rate=0.05, random_seed=42, task_type='GPU', verbose=50)

In [10]:
from sklearn.metrics import roc_auc_score

val_preds = clf.predict_proba(X_test)[:, 1]

auc_score = roc_auc_score(y_test, val_preds)
print(f"Baseline ROC-AUC: {auc_score:.5f}")

Baseline ROC-AUC: 0.96334


In [11]:
feature_imp = pd.Series(clf.get_feature_importance(), index=X_train.columns)
print(feature_imp.sort_values(ascending=False).head(10))

card1             8.100968
C13               7.445857
C1                5.834589
M5                3.374825
M4                2.932595
V283              2.763555
card2             2.722282
P_emaildomain     2.698286
addr1             2.622716
TransactionAmt    2.045916
dtype: float64
